# AI Dataset Studio - Fine-tuning Qwen3 sur Kaggle

Ce notebook evite Google Drive. Il utilise un Dataset Kaggle prive contenant `ai_dataset_studio_finetune_pack.zip`.

Avant de lancer: Settings > Accelerator > GPU.

## 1. Verifier que le GPU est actif

In [ ]:
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA:', torch.version.cuda)
print('cuda=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('STOP: active Settings > Accelerator > GPU, puis relance cette cellule.')

## 2. Correction P100 si necessaire

Si la cellule precedente affiche `Tesla P100` avec un avertissement `sm_60 is not compatible`, lance cette cellule. Ne redemarre pas la session ensuite.

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install --no-cache-dir torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118
!python -c "import torch; print('Torch:', torch.__version__); print('Torch CUDA:', torch.version.cuda); print('cuda=', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"

## 3. Trouver et extraire le paquet

Le fichier zip doit etre ajoute au notebook via le panneau `Input` de Kaggle.

In [ ]:
from pathlib import Path
import zipfile

zip_paths = sorted(Path('/kaggle/input').glob('**/ai_dataset_studio_finetune_pack.zip'))
print('Zip trouves:', [str(p) for p in zip_paths])
assert zip_paths, 'Ajoute le dataset Kaggle contenant ai_dataset_studio_finetune_pack.zip dans Input.'

with zipfile.ZipFile(zip_paths[0]) as archive:
    archive.extractall('/kaggle/working')

required = [
    '/kaggle/working/ai_lab/training/qwen3_sft_train.jsonl',
    '/kaggle/working/ai_lab/training/qwen3_sft_validation.jsonl',
    '/kaggle/working/ai_lab/training/qwen3_sft_test.jsonl',
    '/kaggle/working/ai_lab/fine_tuning/run_ms_swift_sft.sh',
]
for item in required:
    print(item, 'OK' if Path(item).exists() else 'MANQUANT')

## 4. Installer ms-swift

Cette cellule peut prendre plusieurs minutes.

In [ ]:
%cd /kaggle/working
!python -m pip install -U pip
!pip install -U ms-swift transformers
!python -c "import torch; print('Torch apres install:', torch.__version__, torch.version.cuda, torch.cuda.is_available())"

## 5. Mini test rapide

On entraine 12 exemples pour verifier que tout marche avant l'entrainement complet.

In [ ]:
from pathlib import Path

src = Path('/kaggle/working/ai_lab/training/qwen3_sft_train.jsonl')
dst = Path('/kaggle/working/ai_lab/training/qwen3_sft_train_smoke.jsonl')
lines = src.read_text(encoding='utf-8').splitlines()
dst.write_text('\n'.join(lines[:12]) + '\n', encoding='utf-8')
print('Exemples smoke:', min(12, len(lines)))

In [ ]:
%cd /kaggle/working
!DTYPE=float16 EPOCHS=1 TRAIN_DATA=ai_lab/training/qwen3_sft_train_smoke.jsonl VAL_DATA=ai_lab/training/qwen3_sft_validation.jsonl OUTPUT_DIR=ai_lab/fine_tuning/output/qwen3_ai_dataset_studio_lora_smoke EVAL_STEPS=5 SAVE_STEPS=5 GRAD_ACCUM=4 MAX_LENGTH=2048 bash ai_lab/fine_tuning/run_ms_swift_sft.sh

## 6. Entrainement complet

Lance seulement si le mini test a reussi.

In [ ]:
%cd /kaggle/working
!DTYPE=float16 EPOCHS=3 OUTPUT_DIR=ai_lab/fine_tuning/output/qwen3_ai_dataset_studio_lora EVAL_STEPS=20 SAVE_STEPS=20 GRAD_ACCUM=8 MAX_LENGTH=2048 bash ai_lab/fine_tuning/run_ms_swift_sft.sh

## 7. Trouver le dernier checkpoint

Copie-colle la sortie de cette cellule.

In [ ]:
from pathlib import Path

checkpoints = sorted(Path('/kaggle/working/ai_lab/fine_tuning/output').glob('**/checkpoint-*'), key=lambda p: p.stat().st_mtime)
if not checkpoints:
    print('Aucun checkpoint trouve')
else:
    print('Dernier checkpoint:')
    print(checkpoints[-1])